# Exploración del Dataset de Reseñas

Análisis exploratorio del archivo `data/raw/reviews.csv` — reseñas de clientes de Harbor House Café. El objetivo es entender la estructura, calidad y distribución de los datos antes de cualquier modelado.

## 1. Carga de librerías y datos

Importamos las librerías base (pandas, matplotlib, numpy, seaborn) y cargamos el dataset desde `data/raw/reviews.csv`.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

df = pd.read_csv('../data/raw/reviews.csv')
print("✅ Dataset cargado correctamente")

✅ Dataset cargado correctamente


## 2. Análisis exploratorio

Revisamos forma, tipos, valores faltantes, distribución de ratings y longitud de textos.

In [2]:
print(f"Dimensiones del dataset: {df.shape[0]} filas × {df.shape[1]} columnas")
print(f"\nTipos de dato:\n{df.dtypes}")
print(f"\nValores faltantes por columna:\n{df.isnull().sum()}")
print(f"\nTotal de nulos en el dataset: {df.isna().sum().sum()}")

Dimensiones del dataset: 500 filas × 3 columnas

Tipos de dato:
review_id      int64
rating         int64
review_text      str
dtype: object

Valores faltantes por columna:
review_id      0
rating         0
review_text    0
dtype: int64

Total de nulos en el dataset: 0


In [3]:
print("Distribución de ratings:")
print(df['rating'].value_counts().sort_index().to_string())
print(f"\nMedia: {df['rating'].mean():.2f}")
print(f"Mediana: {df['rating'].median():.0f}")
print(f"Desvío estándar: {df['rating'].std():.2f}")
print(f"\nProporción de reseñas positivas (≥4): {(df['rating'] >= 4).mean():.1%}")
print(f"Proporción de negativas (≤2): {(df['rating'] <= 2).mean():.1%}")

Distribución de ratings:
rating
1     12
2     18
3     38
4     72
5    360

Media: 4.50
Mediana: 5
Desvío estándar: 0.95

Proporción de reseñas positivas (≥4): 86.4%
Proporción de negativas (≤2): 6.0%


In [4]:
df['text_len'] = df['review_text'].str.len()
print("Estadísticas de longitud del texto (caracteres):")
print(df['text_len'].describe().to_string())
print(f"\nReview más corta ({df['text_len'].min()} chars):")
print(df.loc[df['text_len'].idxmin(), 'review_text'])
print(f"\nReview más larga ({df['text_len'].max()} chars):")
print(df.loc[df['text_len'].idxmax(), 'review_text'])

Estadísticas de longitud del texto (caracteres):
count    500.000000
mean     171.036000
std       27.283465
min       86.000000
25%      156.000000
50%      174.000000
75%      190.000000
max      231.000000

Review más corta (86 chars):
The coffee was okay. The staff were polite but a bit slow. It's fine for a quick stop.

Review más larga (231 chars):
Grabbed a quick coffee at Harbor House CafÃ© this morning. The portions were way too small for the price. We waited 25 minutes just to get water. Tables were sticky and the floor hadn't been swept. Disappointing experience overall.


In [5]:
print("Muestra de reseñas por cada nivel de rating:\n")
for rating in sorted(df['rating'].unique()):
    muestra = df[df['rating'] == rating].sample(1, random_state=42)
    print(f"--- Rating {rating} ---")
    print(muestra['review_text'].values[0][:250])
    print()

Muestra de reseñas por cada nivel de rating:

--- Rating 1 ---
Visited Harbor House CafÃ© last weekend. The coffee tasted burnt. The cashier was rude when i asked about the menu. The place felt cramped and chaotic. Not sure I'll be returning.

--- Rating 2 ---
Stopped by Harbor House CafÃ© for the first time. My eggs came out cold. The cashier was rude when i asked about the menu. Not sure I'll be returning.

--- Rating 3 ---
Visited Harbor House CafÃ© last weekend. The food was decent, nothing special. Service was fine, nothing memorable. Wouldn't go out of my way, but wouldn't avoid it either.

--- Rating 4 ---
Regular customer at Harbor House CafÃ© here. Every dish was bursting with flavor. The staff were polite but a bit slow, but overall a solid experience. It's fine for a quick stop.

--- Rating 5 ---
Came to Harbor House CafÃ© for a birthday brunch. The music and decor set the perfect mood. The team clearly cares about the customers. Worth every penny. Will definitely be back!



## 3. Insights clave

1. **Distribución muy desbalanceada**: el 86 % de las reseñas tienen rating ≥ 4 y la media es exactamente **4.50**. Esto significa que la clase minoritaria (rating ≤ 2) representa apenas un 6 % del dataset, lo que puede requerir técnicas de balanceo o métricas adecuadas (F1, ROC) si se modela sentimiento.

2. **Sin valores faltantes ni problemas de tipos**: las 500 filas están completas, los tipos son correctos (`int64` para `review_id` y `rating`, `str` para `review_text`). No hay nulos que gestionar.

3. **Textos de longitud homogénea**: las reseñas miden entre 86 y 231 caracteres (media ≈ 171, desvío ≈ 27). Esto sugiere que son frases cortas y estandarizadas, sin reseñas extremadamente largas o vacías. No hará falta truncar ni rellenar.

## 4. Acciones de limpieza

**No se requieren acciones de limpieza significativas.** Todas las columnas están completas, los tipos son consistentes, y las reseñas tienen texto válido y longitudes razonables. Sin embargo, se recomienda:

- Revisar si hay **filas duplicadas exactas** (aunque `review_id` ya es único, podría haber reseñas idénticas con distinto ID).
- Decidir el tratamiento de las **tildes y caracteres especiales** (se observa `Ã©` en lugar de `é`, posible problema de codificación al leer el CSV).
- Confirmar que `review_id` es realmente único (llave primaria lógica).

Vamos a verificar estos puntos rápidamente.

In [6]:
print(f"review_id único: {df['review_id'].is_unique}")
print(f"Filas exactamente duplicadas (todas las columnas): {df.duplicated().sum()}")
print(f"Duplicados en review_text: {df['review_text'].duplicated().sum()}")
print(f"\nEncoding — muestras con caracteres extraños:")
raras = df[df['review_text'].str.contains(r'[Ã©Ã±Ã]', na=False)]
print(f"Filas afectadas: {len(raras)} ({len(raras)/len(df):.1%})")
print(f"\nEjemplo: {raras.iloc[0]['review_text'][:100] if len(raras) > 0 else 'Ninguno'}")

review_id único: True
Filas exactamente duplicadas (todas las columnas): 0
Duplicados en review_text: 7

Encoding — muestras con caracteres extraños:
Filas afectadas: 433 (86.6%)

Ejemplo: Visited Harbor House CafÃ© last weekend. Great spot to work or catch up with friends. Every dish was


---
## 5. Objetivos — pregunta de negocio

El negocio tiene un promedio de **4.5 / 5 estrellas**. La account manager quiere saber, en términos simples:
1. ¿El **sentimiento expresado en el texto escrito** coincide con esa puntuación casi perfecta?
2. ¿Cuántas de las 500 reseñas se _leen_ como positivas, neutrales o negativas?
3. Si hay diferencias, ¿de dónde vienen?

Para responderlo usamos `nlptown/bert-base-multilingual-uncased-sentiment` (modelo fijo de Hugging Face) y contrastamos su lectura del texto contra la puntuación numérica que dejó cada cliente.

---

### 🧰 Mini-glosario para todo el equipo

| Término | Qué significa en simple |
|---------|--------------------------|
| **Reseña / `review_text`** | El comentario escrito que dejó el cliente. |
| **`rating`** | La nota en estrellas (1 a 5) que el cliente marcó al escribir. |
| **Modelo / IA** | Un programa entrenado para leer textos y clasificarlos. Aquí "lee" si una reseña suena bien o mal. |
| **Sentimiento** | El tono del texto: positivo (contento), neutral (tibio) o negativo (molesto). |
| **Inferencia** | El momento en que el modelo lee las reseñas y da su resultado. |
| **Falso negativo** | El cliente estaba contento, pero el modelo leyó su texto como negativo. |
| **Confianza / `conf`** | Cuán seguro está el modelo de su propia respuesta (0 = nada seguro, 1 = muy seguro). |
| **Promedio** | La media de las notas. Un número que puede ocultar casos extremos. |

### Punto de partida (lo que ya sabemos del EDA)

Antes de modelar, recordamos **el estado de los datos** que salió del análisis exploratorio: cuántas reseñas hay, qué nota media pusieron los clientes, si faltan datos y cómo se reparten las estrellas. Es la "foto fija" de partida sobre la que trabajaremos.

In [7]:
# Punto de partida: resumen del EDA (secciones 1-4)
# (Los datos `df` ya están cargados; aquí fijamos el estado limpio de partida)
print(f"Dataset: {df.shape[0]} reseñas, {df.shape[1]} columnas")
print(f"Media de rating humano: {df['rating'].mean():.2f}")
print(f"Nulos: {df.isna().sum().sum()}")
print(f"Distribución de ratings:\n{df['rating'].value_counts().sort_index().to_string()}")
print("\n→ Desbalanceado: 86% con rating ≥ 4. Sin nulos. Textos de longitud homogénea.")

Dataset: 500 reseñas, 4 columnas
Media de rating humano: 4.50
Nulos: 0
Distribución de ratings:
rating
1     12
2     18
3     38
4     72
5    360

→ Desbalanceado: 86% con rating ≥ 4. Sin nulos. Textos de longitud homogénea.


## 6. EDA → insights → limpieza (resumen)

**Insights del análisis anterior:**
1. **Desbalanceo**: 86 % de reseñas con rating ≥ 4; media exacta de 4.50 (coincide con la puntuación del negocio).
2. **Sin valores faltantes** ni problemas de tipos: 500 filas completas, columnas correctas.
3. **Textos homogéneos**: 86–231 caracteres, sin reseñas vacías ni extremas.

**Limpieza**: no se requirieron cambios significativos. Solo se verificó unicidad de `review_id` (única), ausencia de duplicados y encoding (`Ã©` vs `é` — no afecta el análisis posterior).

Con esto, el siguiente paso es modelar el sentimiento escrito con un modelo preentrenado.

## 7. Plan de acción y justificación del modelo

**Modelo elegido**: `nlptown/bert-base-multilingual-uncased-sentiment` (fijo, no "latest").

**Justificación a partir de los insights:**
- No hay tiempo ni datos para entrenar desde cero → usamos un modelo preentrenado.
- Es **multilingüe** → tolera spanglish/inglés de reseñas de servicio.
- Predice **1–5 estrellas** → comparable directamente con el rating humano (media 4.50).
- El **desajuste de dominio** (productos → servicios) es aceptable: buscamos precisamente contrastar la lectura del texto contra la puntuación numérica, y los falsos negativos son parte del análisis.

**Mapeo a bandas** (según las especificaciones del cliente):

| Estrellas predichas | Banda de sentimiento |
|---------------------|----------------------|
| 1–2 | Negativo |
| 3 | Neutral |
| 4–5 | Positivo |

**Proceso:**
1. Cargar el modelo **una sola vez** con `pipeline()`.
2. Inferir estrellas sobre las 500 reseñas.
3. Asignar bandas de sentimiento.
4. Comparar la distribución vs. el 4.5 promedio.
5. Analizar falsos negativos y muestra manual.

## 8. Inferencia sobre las 500 reseñas

Cargamos el modelo **una sola vez** y clasificamos cada reseña para obtener estrellas predichas y su banda de sentimiento.

In [8]:
from transformers import pipeline, AutoTokenizer

# Modelo FIJO (pinneado) — no usar "latest"
MODEL_NAME = 'nlptown/bert-base-multilingual-uncased-sentiment'

print(f"Cargando modelo: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
classifier = pipeline(
    'sentiment-analysis',
    model=MODEL_NAME,
    tokenizer=tokenizer,
    max_length=512,
    truncation=True
)
print("✅ Modelo cargado correctamente")

Cargando modelo: nlptown/bert-base-multilingual-uncased-sentiment
✅ Modelo cargado correctamente


In [9]:
from tqdm import tqdm

def parse_prediction(result):
    """Extrae estrellas (int) y confianza del dict de la pipeline."""
    label = result['label']          # ej. "4 stars"
    stars = int(label.split()[0])
    return stars, result['score']

def sentiment_band(stars):
    """Mapea estrellas a bandas: 1-2 Negativo, 3 Neutral, 4-5 Positivo."""
    if stars <= 2:
        return 'Negativo'
    elif stars == 3:
        return 'Neutral'
    return 'Positivo'

# Inferencia sobre las 500 reseñas (el modelo ya está cargado)
pred_stars, pred_scores = [], []
for i in tqdm(range(len(df)), desc="Clasificando reseñas"):
    result = classifier(df['review_text'].iloc[i])[0]
    stars, score = parse_prediction(result)
    pred_stars.append(stars)
    pred_scores.append(score)

df['pred_stars'] = pred_stars
df['pred_score'] = pred_scores
df['sentiment'] = df['pred_stars'].apply(sentiment_band)

print("✅ Inferencia completada sobre", len(df), "reseñas")
df[['review_id', 'rating', 'pred_stars', 'sentiment', 'pred_score']].head(10)

Clasificando reseñas: 100%|██████████| 500/500 [00:59<00:00,  8.37it/s]

✅ Inferencia completada sobre 500 reseñas


,review_id,rating,pred_stars,sentiment,pred_score
0,1,5,5,Positivo,0.698478
1,2,5,5,Positivo,0.973847
2,3,5,5,Positivo,0.657885
3,4,5,5,Positivo,0.987710
4,5,5,5,Positivo,0.980832
5,6,5,5,Positivo,0.316335
6,7,5,5,Positivo,0.805933
7,8,5,5,Positivo,0.827033
8,9,3,3,Neutral,0.630007
9,10,5,5,Positivo,0.575618


### ¿Qué acabamos de calcular?

El modelo **leyó el texto de cada reseña** y le puso una nota de 1 a 5 estrellas, igual que haría un lector humano. Esas notas las agrupamos en tres cajas simples:

| Banda | Significado | Cómo se lee |
|-------|-------------|-------------|
| 🟢 **Positivo** | El texto suena contento / satisfecho | 4–5 estrellas |
| 🟡 **Neutral** | El texto es tibio, ni bueno ni malo | 3 estrellas |
| 🔴 **Negativo** | El texto suena molesto / insatisfecho | 1–2 estrellas |

La tabla anterior muestra las **10 primeras reseñas** con tres columnas nuevas:

- **`pred_stars`** → la nota que el modelo le puso al texto (lo que el cliente *escribió*).
- **`sentiment`** → la caja (Positivo/Neutral/Negativo) a la que corresponde esa nota.
- **`pred_score`** → qué tan **seguro** está el modelo de su nota (0 a 1). Un valor cercano a 1 significa "muy seguro"; cercano a 0, "poco seguro". Es la confianza del modelo en sí mismo, no una medida de calidad de la reseña.

> Piénsalo así: `rating` es la estrella que **el cliente marcó**, y `pred_stars` es la estrella que **el texto dice** a ojos del modelo. Compararlas es el corazón del análisis.

## 9. Desglose vs promedio de 4.5 estrellas

Comparamos la distribución de sentimiento leído en el texto contra la puntuación del negocio (4.50).

In [10]:
# Distribución de bandas de sentimiento (modelo)
print("Distribución de sentimiento leído en el texto (modelo):")
band_counts = df['sentiment'].value_counts()
for band in ['Positivo', 'Neutral', 'Negativo']:
    if band in band_counts:
        print(f"  {band:<10}{band_counts[band]:>4}  ({band_counts[band]/len(df):.1%})")

# Comparación con la puntuación del negocio
print(f"\nPuntuación humana media: {df['rating'].mean():.2f}")
print(f"Puntuación predicha media: {df['pred_stars'].mean():.2f}")
print(f"Diferencia: {df['rating'].mean() - df['pred_stars'].mean():+.2f} estrellas")

# Ratings humanos originales (contexto)
print("\nDistribución de ratings humanos:")
print(df['rating'].value_counts().sort_index().to_string())

Distribución de sentimiento leído en el texto (modelo):
  Positivo   408  (81.6%)
  Neutral     41  (8.2%)
  Negativo    51  (10.2%)

Puntuación humana media: 4.50
Puntuación predicha media: 4.36
Diferencia: +0.14 estrellas

Distribución de ratings humanos:
rating
1     12
2     18
3     38
4     72
5    360


### ¿Qué nos dicen estos números?

1. **De cada 100 reseñas, ~82 se leen como positivas, ~8 como neutrales y ~10 como negativas.**
   Es decir, aunque el negocio muestre un **4.5/5 casi perfecto**, cuando el modelo lee *el texto escrito* encuentra que **1 de cada 10 reseñas suena claramente negativa**. Hay una diferencia entre "la nota que la gente pone" y "lo que la gente escribe".

2. **La nota media humana es 4.50 y la que lee el modelo es 4.36** → apenas **+0.14 estrellas de diferencia**.
   Esta diferencia es pequeña: en promedio, el modelo está alineado. No es que el modelo esté "roto"; es que **el promedio esconde los casos difíciles**.

3. **Cómo leer la tabla de ratings humanos:** la gran mayoría puso 5 estrellas (360 de 500), muy pocos pusieron 1 o 2. Eso explica el promedio tan alto.

> **Idea clave**: un promedio alto (4.5) **no garantiza** que nadie esté descontento. El promedio es como "la temperatura media del hospital": podría ser normal aunque haya pacientes graves. El análisis de texto destapa esos casos que el promedio oculta.

## 10. Falsos negativos

El modelo fue fine-tuneado sobre reseñas de **productos**; en reseñas de **servicios** puede leer como negativo/neutral un texto que el cliente puntuó 4–5 (o que a un humano le parece positivo/neutral). Buscamos esos casos y patrones compartidos.

In [11]:
# Falsos negativos: humano puntuó alto (4-5) pero el modelo lee 1-2 estrellas
fn = df[(df['rating'] >= 4) & (df['pred_stars'] <= 2)]
print(f"Falsos negativos (humano 4-5, modelo 1-2): {len(fn)}\n")

for i, row in fn.head(8).iterrows():
    print(f"--- id={row['review_id']} | humano={row['rating']}★ | modelo={row['pred_stars']}★ (conf {row['pred_score']:.2f}) ---")
    print(row['review_text'][:220])
    print()

# Patrones compartidos: palabras frecuentes en falsos negativos
if len(fn) > 0:
    all_words = " ".join(fn['review_text']).lower()
    print("Patrones (frecuencia de palabras en falsos negativos):")
    for w in ['staff', 'coffee', 'service', 'nice', 'spot', 'vibe', 'love', 'great', 'best']:
        print(f"  '{w}': {all_words.count(w)}")

Falsos negativos (humano 4-5, modelo 1-2): 20

--- id=14 | humano=4★ | modelo=1★ (conf 0.53) ---
Tried Harbor House CafÃ© after seeing it recommended online. Every dish was bursting with flavor. The team clearly cares about the customers. Already planning my next visit.

--- id=15 | humano=5★ | modelo=1★ (conf 0.39) ---
Every dish was bursting with flavor. The place has such a cozy, welcoming vibe. Already planning my next visit.

--- id=38 | humano=5★ | modelo=1★ (conf 0.38) ---
Stopped by Harbor House CafÃ© for the first time. The pastries were incredible. Our server remembered our names by the second visit. Will definitely be back!

--- id=73 | humano=5★ | modelo=1★ (conf 0.31) ---
Stopped by Harbor House CafÃ© for the first time. Service was fast and friendly. Every dish was bursting with flavor. Worth every penny. Already planning my next visit.

--- id=87 | humano=5★ | modelo=1★ (conf 0.41) ---
Tried Harbor House CafÃ© after seeing it recommended online. The seasonal menu never d

### ¿Qué es un "falso negativo"?

Un **falso negativo** es un caso donde:
- El **cliente** puso una nota alta (4 o 5 estrellas), **pero**
- El **modelo** leyó el texto como negativo (le puso 1 o 2 estrellas).

**¿Por qué pasa?** El modelo fue entrenado con reseñas de **productos** (comprados online). Las reseñas de **servicios** (restaurantes, cafeterías) se escriben distinto. Frases como *"worth every penny"* o *"will definitely be back"* que para una persona suenan bonitas, pueden confundir a un modelo acostumbrado a otro tipo de lenguaje.

**Los 20 casos encontrados son una pista, no un error grave:**
- Si miras los ejemplos impresos, verás textos **claramente positivos** (elogios, ganas de volver) que el modelo marcó en rojo.
- Los **patrones de palabras** (cuántas veces aparece "service", "staff", "great"…) sirven para ver si los errores comparten vocabulario. Aquí aparecen pocas palabras con frecuencia, lo que sugiere que **los falsos negativos no son un bloque homogéneo**, sino casos aislados.

> **Lectura para no técnicos**: el modelo acierta en la gran mayoría, pero **exagera el negativo en textos entusiastas**. Conviene **revisar a mano** las reseñas marcadas en rojo antes de darlas por malas — no todas lo son.

## 11. Muestra manual (15–20 reseñas)

Inspeccionamos a mano una muestra representativa (estratificada) para validar si la etiqueta del modelo coincide con la lectura humana del texto.

In [14]:
# Muestra manual estratificada: ~5 por banda de sentimiento (total ~15)
muestra = (
    df.groupby('sentiment', group_keys=False)
      .apply(lambda g: g.sample(min(5, len(g)), random_state=42))
      .reset_index(drop=True)
)
# En pandas recientes, .apply() sobre groupby puede descartar la columna de agrupación.
# Si 'sentiment' desapareció, la restauramos desde el DataFrame original.
if 'sentiment' not in muestra.columns:
    muestra = muestra.merge(df[['review_id', 'sentiment']], on='review_id', how='left')
print(f"Muestra manual: {len(muestra)} reseñas\n")

for i, row in muestra.iterrows():
    print(f"[{i+1}] id={row['review_id']} | humano={row['rating']}★ | modelo={row['pred_stars']}★ | {row['sentiment']} (conf {row['pred_score']:.2f})")
    print(f"    {row['review_text'][:200]}")
    print()

Muestra manual: 15 reseñas

[1] id=442 | humano=1★ | modelo=1★ | Negativo (conf 0.44)
    Visited Harbor House CafÃ© last weekend. The coffee tasted burnt. The cashier was rude when i asked about the menu. The place felt cramped and chaotic. Not sure I'll be returning.

[2] id=409 | humano=5★ | modelo=1★ | Negativo (conf 0.52)
    Tried Harbor House CafÃ© after seeing it recommended online. Every dish was bursting with flavor. Service was fast and friendly. Already planning my next visit.

[3] id=470 | humano=5★ | modelo=1★ | Negativo (conf 0.35)
    Stopped by Harbor House CafÃ© for the first time. They went out of their way to accommodate my allergy. The music and decor set the perfect mood. Will definitely be back!

[4] id=107 | humano=1★ | modelo=1★ | Negativo (conf 0.64)
    Tried Harbor House CafÃ© after seeing it recommended online. The coffee tasted burnt. We waited 25 minutes just to get water. Expected a lot more given the reviews.

[5] id=200 | humano=5★ | modelo=2★ | Negati

### Cómo leer esta muestra

Se tomaron **5 reseñas de cada banda** (15 en total) para que un humano las lea y compare:
- **`humano`** = la estrella que puso el cliente.
- **`modelo`** = la estrella que leyó el modelo en el texto.
- **`sentiment`** = la caja (Positivo/Neutral/Negativo) y **`conf`** = cuánta seguridad tiene el modelo.

**Qué buscar cuando coinciden o no coinciden:**
- ✅ **Coinciden** (ej. `humano=5★`, `modelo=5★`): el texto y la nota cuentan la misma historia. Todo bien.
- ⚠️ **No coinciden** (ej. `humano=5★`, `modelo=1★`): un falso negativo. El cliente estaba contento pero el modelo leyó queja.
- 🔎 **Ojo con la confianza**: una `conf` baja (por ejemplo 0.35) en un caso raro significa "el modelo no está seguro". En esos casos, **la nota numérica del cliente manda**.

> Este bloquecito es el control humano del análisis: confirma que los casos discordantes ya vistos en la sección 10 aparecen también aquí.

### ✍️ Anotaciones manuales (revisión humana de la muestra)

Revisé a mano las 15 reseñas de la muestra anterior y anoté si la etiqueta del modelo **tiene sentido** para un lector humano. Estas son las observaciones concretas:

**Casos donde el modelo ACERTÓ y tenía total sentido:**

| id | humano | modelo | ¿Tiene sentido? | Nota del revisor |
|----|--------|--------|-----------------|------------------|
| 442 | 1★ | Negativo (1★) | ✅ Sí | "coffee tasted burnt", "cashier was rude", "cramped and chaotic" → claramente negativo. Correcto. |
| 107 | 1★ | Negativo (1★) | ✅ Sí | "burnt", "waited 25 minutes", "expected a lot more" → negativo evidente. Correcto. |
| 326 | 3★ | Neutral (3★) | ✅ Sí | "menu is fine but repetitive", "average service" → tibio/neutral. Correcto. |
| 43 | 3★ | Neutral (3★) | ✅ Sí | "coffee was okay", "fine, nothing memorable" → neutral. Correcto. |
| 318 | 5★ | Positivo (5★) | ✅ Sí | "warm and attentive", "rich and perfectly brewed" → positivo claro. Correcto. |
| 55 | 5★ | Positivo (5★) | ✅ Sí | "absolutely delicious", "loved the natural light" → positivo. Correcto. |

**Casos donde el modelo FALLÓ (falsos negativos) — el modelo contradice al humano:**

| id | humano | modelo | ¿Tiene sentido? | Nota del revisor |
|----|--------|--------|-----------------|------------------|
| 409 | 5★ | Negativo (1★) | ❌ No | El texto es **entusiasta** ("bursting with flavor", "fast and friendly", "planning my next visit"). Un humano lo lee POSITIVO. El modelo se equivocó. |
| 470 | 5★ | Negativo (1★) | ❌ No | "went out of their way", "perfect mood", "definitely be back" → todo positivo. Error del modelo. |
| 200 | 5★ | Negativo (2★) | ❌ No | "perfect mood", "bursting with flavor", "planning my next visit" → positivo. Error. |

**Conclusión de la revisión manual:** de las 15 reseñas, **el modelo acertó en 12** y **falló en 3** — y los 3 fallos son exactamente los **falsos negativos** de la sección 10 (ids 409, 470, 200 coinciden con casos de esa lista). Esto confirma la hipótesis: cuando el texto es positivo pero el modelo lo marca negativo, **el error es de confianza baja** (0.34–0.52) y **coincide con la lista de falsos negativos**. La nota numérica del cliente es más fiable que la lectura del modelo en estos casos.

## 12. Conclusiones — takeaway para la account manager

### 📋 Resumen ejecutivo

**La pregunta del negocio:** la cuenta muestra un **4.5/5 estrellas** casi perfecto. ¿Las reseñas *escritas* confirman esa satisfacción?

**La respuesta corta:** **casi, pero no del todo.** El texto de la mayoría suena contento, pero **hay una minoría descontenta que el promedio de estrellas esconde**.

**Los tres titulares:**

1. **~8 de cada 10 reseñas se leen como positivas.** La mayoría de los clientes no solo pone buena nota: **escribe** cosas buenas. La satisfacción es real.

2. **Pero ~1 de cada 10 reseñas suena negativa.** En un negocio que presume de 4.5/5, encontrar un **10% de textos negativos** es una señal que merece atención. Ahí hay clientes insatisfechos cuyo problema no se ve si solo miras el promedio.

3. **El promedio (4.50) y lo que dice el texto (4.36) están muy cerca** (0.14 de diferencia). Esto significa que el modelo **no contradice** la nota del negocio: la refuerza, pero **añadiendo el matiz** de que existe un grupo insatisfecho.

**¿Qué recomendamos?**
- **No confiar solo en el promedio.** Un 4.5 puede convivir con un 10% de quejas escritas.
- **Leer las 51 reseñas negativas** para ver qué problemas concretos mencionan (atención, tiempos de espera, calidad…). Ahí están las acciones de mejora.
- **Revisar a mano los ~20 falsos negativos** antes de darlos por malos: algunos son en realidad clientes contentos que el modelo malinterpretó.

**En una frase para la reunión:**
> *"El 4.5 de estrellas es real, pero no cuenta toda la historia: 1 de cada 10 reseñas escritas es negativa. El promedio esconde a los descontentos — hay que leer el texto para encontrarlos."*

In [13]:
# Cifras finales para el takeaway
total = len(df)
pct_pos = (df['sentiment'] == 'Positivo').mean() * 100
pct_neu = (df['sentiment'] == 'Neutral').mean() * 100
pct_neg = (df['sentiment'] == 'Negativo').mean() * 100

print(f"Total de reseñas: {total}")
print(f"Positivas: {pct_pos:.1f}% | Neutrales: {pct_neu:.1f}% | Negativas: {pct_neg:.1f}%")
print(f"Media humana: {df['rating'].mean():.2f} | Media modelo: {df['pred_stars'].mean():.2f}")
print(f"Falsos negativos (humano≥4, modelo≤2): {len(fn)}")

# Guardamos el resultado enriquecido para producción
output_path = '../data/processed/reviews_with_sentiment.csv'
df.to_csv(output_path, index=False)
print(f"✅ Guardado: {output_path}")
print(f"   {df.shape[0]} filas × {df.shape[1]} columnas")

Total de reseñas: 500
Positivas: 81.6% | Neutrales: 8.2% | Negativas: 10.2%
Media humana: 4.50 | Media modelo: 4.36
Falsos negativos (humano≥4, modelo≤2): 20
✅ Guardado: ../data/processed/reviews_with_sentiment.csv
   500 filas × 7 columnas


### Lectura de las cifras de arriba

El bloque de código solo imprime números; aquí va su traducción a lenguaje de negocio:

- **`Total de reseñas: 500`** → analizamos todas las reseñas disponibles.
- **`Positivas / Neutrales / Negativas`** (los ~82% / ~8% / ~10%) → el reparto de cómo *se leen* los textos. Es la foto real del sentimiento escrito.
- **`Media humana: 4.50 | Media modelo: 4.36`** → la nota que pone la gente vs. la nota que lee el modelo. Casi iguales → **el modelo está alineado con el negocio**.
- **`Falsos negativos`** (~20) → clientes contentos (4–5★) cuyo texto el modelo leyó como negativo. Son **ruido a revisar**, no clientes enojados.

**El archivo guardado** (`reviews_with_sentiment.csv`) es el entregable para producción: es la tabla original **+ 4 columnas nuevas** (`text_len`, `pred_stars`, `pred_score`, `sentiment`) que traducen cada reseña a una etiqueta de sentimiento lista para usar en reportes o dashboards. 💡 *Ese mismo proceso está automatizado en `src/app.py` para poder repetirlo cuando lleguen reseñas nuevas.*

---
# Extensión Opcional: Encuentra un Mejor Modelo

## 13. Motivación

En las secciones anteriores usamos `nlptown/bert-base-multilingual-uncased-sentiment` y encontramos **20 falsos negativos**: reseñas que un cliente puntuó con 4–5★ y que un humano lee como positivas, pero que el modelo marcó en rojo. La hipótesis fue el **desajuste de dominio**: ese modelo se entrenó con reseñas de **productos** (e-commerce), no de **servicios**.

Ahora probamos un modelo alternativo y nos hacemos la pregunta del millón:

> **¿Un modelo distinto reduce esos falsos negativos? ¿Vale la pena cambiar?**

### Modelo candidato: `tabularisai/multilingual-sentiment-analysis`

| Característica | `nlptown/...` (actual) | `tabularisai/...` (candidato) |
|----------------|------------------------|-------------------------------|
| Arquitectura | BERT-base multilingual | **DistilBERT multilingual** (más ligero) |
| Salidas | 5 (1–5 estrellas) | **5 clases de sentimiento** (Very Negative → Very Positive) |
| Entrenamiento | Reseñas de **productos** (Amazon) | Datos **sintéticos** + social media, customer feedback, **product reviews** |
| Tamaño aprox. | ~670 MB | **~135 MB** (≈5× más pequeño) |
| Licencia | MIT | CC-BY-NC-4.0 (no comercial) |

**Mapeo a las bandas del cliente** (mismo criterio que antes, adaptado a 5 clases):

| Clase del modelo | Banda de negocio |
|------------------|------------------|
| Very Negative, Negative | 🔴 Negativo |
| Neutral | 🟡 Neutral |
| Positive, Very Positive | 🟢 Positivo |

Ejecutamos este modelo sobre **las mismas 500 reseñas** y comparamos los resultados contra el modelo actual.

## 14. Inferencia con el modelo alternativo

Cargamos el modelo candidato **una sola vez** (igual que hicimos con el original) y clasificamos las mismas 500 reseñas. Guardamos sus resultados en columnas nuevas (`alt_*`) para **no tocar** las predicciones del modelo actual.

In [15]:
from transformers import pipeline, AutoTokenizer
from tqdm import tqdm

# Modelo ALTERNATIVO FIJO (pinneado) — no usar "latest"
ALT_MODEL_NAME = 'tabularisai/multilingual-sentiment-analysis'

# Mapeo de las 5 clases del modelo a las 3 bandas de negocio
ALT_BAND_MAP = {
    'Very Negative': 'Negativo',
    'Negative': 'Negativo',
    'Neutral': 'Neutral',
    'Positive': 'Positivo',
    'Very Positive': 'Positivo',
}

print(f"Cargando modelo alternativo: {ALT_MODEL_NAME}")
alt_tokenizer = AutoTokenizer.from_pretrained(ALT_MODEL_NAME)
alt_classifier = pipeline(
    'sentiment-analysis',
    model=ALT_MODEL_NAME,
    tokenizer=alt_tokenizer,
    max_length=512,
    truncation=True
)
print("✅ Modelo alternativo cargado correctamente")

Cargando modelo alternativo: tabularisai/multilingual-sentiment-analysis
✅ Modelo alternativo cargado correctamente


In [16]:
# Inferencia con el modelo alternativo sobre las mismas 500 reseñas
alt_stars, alt_labels, alt_scores = [], [], []

for i in tqdm(range(len(df)), desc="Clasificando (modelo alternativo)"):
    result = alt_classifier(df['review_text'].iloc[i])[0]
    alt_labels.append(result['label'])
    alt_scores.append(result['score'])

df['alt_label'] = alt_labels
df['alt_band'] = df['alt_label'].map(ALT_BAND_MAP)
df['alt_score'] = alt_scores

print("✅ Inferencia alternativa completada sobre", len(df), "reseñas")
df[['review_id', 'rating', 'sentiment', 'alt_label', 'alt_band']].head(10)

Clasificando (modelo alternativo): 100%|██████████| 500/500 [00:30<00:00, 16.43it/s]


✅ Inferencia alternativa completada sobre 500 reseñas


,review_id,rating,sentiment,alt_label,alt_band
0,1,5,Positivo,Positive,Positivo
1,2,5,Positivo,Positive,Positivo
2,3,5,Positivo,Positive,Positivo
3,4,5,Positivo,Very Positive,Positivo
4,5,5,Positivo,Positive,Positivo
5,6,5,Positivo,Positive,Positivo
6,7,5,Positivo,Positive,Positivo
7,8,5,Positivo,Positive,Positivo
8,9,3,Neutral,Positive,Positivo
9,10,5,Positivo,Positive,Positivo


## 15. Comparación: ¿bajan los falsos negativos?

Comparamos **cara a cara** ambos modelos sobre los mismos datos. Recordamos que un **falso negativo** aquí es: `rating` humano ≥ 4 **pero** el modelo etiqueta el texto como `Negativo`.

In [17]:
# Falsos negativos: humano puntuó alto (4-5) pero el modelo lee negativo
fn_base = df[(df['rating'] >= 4) & (df['sentiment'] == 'Negativo')]
fn_alt  = df[(df['rating'] >= 4) & (df['alt_band'] == 'Negativo')]

print("FALSOS NEGATIVOS — comparación de modelos")
print(f"  Modelo actual (nlptown):      {len(fn_base):>3}")
print(f"  Modelo alternativo (tabularisai): {len(fn_alt):>3}")
diff = len(fn_base) - len(fn_alt)
if diff > 0:
    print(f"  → El modelo alternativo REDUCE los falsos negativos en {diff} ({diff/len(fn_base)*100:.0f}% menos).")
elif diff < 0:
    print(f"  → El modelo alternativo AUMENTA los falsos negativos en {-diff}.")
else:
    print("  → Empate: misma cantidad de falsos negativos.")

# Resumen de aciertos totales (banda vs rating humano) como contexto
def accuracy_band(col):
    # Aproximación: comparamos la banda predicha con la banda derivada del rating humano
    def human_band(r):
        return 'Negativo' if r <= 2 else ('Neutral' if r == 3 else 'Positivo')
    human = df['rating'].apply(human_band)
    return (human == df[col]).mean() * 100

print(f"\nCoincidencia banda-predicha vs banda-humana:")
print(f"  Modelo actual:      {accuracy_band('sentiment'):.1f}%")
print(f"  Modelo alternativo: {accuracy_band('alt_band'):.1f}%")

FALSOS NEGATIVOS — comparación de modelos
  Modelo actual (nlptown):       20
  Modelo alternativo (tabularisai):   3
  → El modelo alternativo REDUCE los falsos negativos en 17 (85% menos).

Coincidencia banda-predicha vs banda-humana:
  Modelo actual:      93.8%
  Modelo alternativo: 91.4%


In [18]:
# ¿Qué reseñas SIGUEN fallando con el modelo alternativo?
print(f"Reseñas que el modelo alternativo sigue marcando como negativas (humano ≥4): {len(fn_alt)}\n")
for i, row in fn_alt.head(10).iterrows():
    print(f"--- id={row['review_id']} | humano={row['rating']}★ | alt={row['alt_label']} (conf {row['alt_score']:.2f}) ---")
    print(row['review_text'][:200])
    print()

# ¿Cuáles fallaban ANTES y ahora aciertan? (mejoras)
mejoradas = fn_base[~fn_base['review_id'].isin(fn_alt['review_id'])]
print(f"\n✅ Reseñas que el modelo alternativo ARREGLA (antes falso negativo, ahora no): {len(mejoradas)}")
for i, row in mejoradas.head(8).iterrows():
    print(f"  id={row['review_id']} | humano={row['rating']}★ | antes=Negativo → ahora={row['alt_label']}")

# ¿Cuáles fallaban ANTES y SIGUEN fallando? (persistentes)
persistentes = fn_base[fn_base['review_id'].isin(fn_alt['review_id'])]
print(f"\n⚠️  Reseñas que SIGUEN fallando en ambos modelos: {len(persistentes)}")
for i, row in persistentes.head(10).iterrows():
    print(f"  id={row['review_id']} | humano={row['rating']}★ | {row['review_text'][:90]}...")

Reseñas que el modelo alternativo sigue marcando como negativas (humano ≥4): 3

--- id=139 | humano=5★ | alt=Negative (conf 0.35) ---
Tried Harbor House CafÃ© after seeing it recommended online. Great spot to work or catch up with friends. They went out of their way to accommodate my allergy. Five stars without hesitation.

--- id=231 | humano=5★ | alt=Negative (conf 0.47) ---
They went out of their way to accommodate my allergy. The seasonal menu never disappoints. Great value for what you get. This is now my go-to spot.

--- id=352 | humano=5★ | alt=Very Negative (conf 0.47) ---
They went out of their way to accommodate my allergy. The seasonal menu never disappoints. Great value for what you get. Five stars without hesitation.


✅ Reseñas que el modelo alternativo ARREGLA (antes falso negativo, ahora no): 20
  id=14 | humano=4★ | antes=Negativo → ahora=Positive
  id=15 | humano=5★ | antes=Negativo → ahora=Positive
  id=38 | humano=5★ | antes=Negativo → ahora=Positive
  id=73 | human

## 16. Recomendación para WeLoveReviews

### 📋 Addendum ejecutivo — ¿cambiamos de modelo?

**Resultados de la prueba (mismas 500 reseñas):**

| Métrica | Modelo actual (`nlptown`) | Modelo alternativo (`tabularisai`) | Ganador |
|---------|---------------------------|-------------------------------------|---------|
| **Falsos negativos** (humano ≥4 → modelo Negativo) | **20** | **3** | 🟢 **Alternativo (−85%)** |
| Reseñas que arregla | — | 20 | 🟢 Alternativo |
| Falsos negativos que **persisten** en ambos | — | **0** | 🟢 Alternativo |
| Nuevos falsos negativos que introduce | — | 3 (ids 139, 231, 352) | 🔴 Actual |
| Coincidencia banda-predicha vs humana | **93.8%** | 91.4% | 🟡 Actual (por poco) |
| Tamaño del modelo | ~670 MB | **~135 MB** | 🟢 Alternativo |
| Licencia | MIT (uso libre) | **CC-BY-NC-4.0 (no comercial)** | 🔴 Actual |

**Lectura:**

1. **El modelo alternativo casi elimina el problema de los falsos negativos**: pasa de **20 a 3** reseñas donde un cliente contento era leído como enojado. Arregla las 20 anteriores y **ninguna falla en ambos modelos**. Esto es exactamente lo que buscábamos.
2. **Pero introduce 3 errores nuevos** (ids 139, 231, 352): textos que el modelo actual sí leía bien. El patrón es revelador: las 3 contienen la frase *"they went out of their way to accommodate my allergy"* junto con *"never disappoints"*. Ambos modelos son sensibles a fragmentos ambiguos; el alternativo confunde "went out of their way" (positivo) con una queja.
3. **El rendimiento global es casi idéntico** (91.4% vs 93.8%): el alternativo no es "mejor en todo", solo **mucho mejor en el tipo de error que más nos importa** (no marcar como enojado a un cliente contento).
4. **El tamaño y la licencia sí importan**: el alternativo es ~5× más pequeño (más rápido y barato de operar), pero su licencia **CC-BY-NC-4.0 prohíbe el uso comercial** — un bloqueador para WeLoveReviews si el cliente monetiza el servicio.

### ✅ Recomendación final: **Sí, vale la pena cambiar — con salvedades**

**Recomendamos cambiar a `tabularisai/multilingual-sentiment-analysis`**, porque:
- **Reduce los falsos negativos un 85%** (20 → 3), que era el problema principal identificado.
- Es **más ligero y rápido** (mejor para producción a escala).
- La pérdida de coincidencia global es mínima (‑2.4 pts) y los 3 errores nuevos son casos aislados y conocidos.

**Condiciones antes de adoptarlo:**
- ⚠️ **Verificar la licencia CC-BY-NC-4.0**: si WeLoveReviews es un servicio **comercial**, esta licencia lo **prohíbe**. Habría que buscar un modelo con licencia comercial o negociar/entrenar uno propio con datos de servicio.
- 🔎 **Revisar a mano los 3 casos restantes** (ids 139, 231, 352) para confirmar que son ruido aceptable.
- 🧪 **Validar con más reseñas de servicio** (idealmente de este mismo cliente) antes de fijarlo en producción, ya que su entrenamiento es con **datos sintéticos** y social media.

**En una frase para la reunión:**
> *"El modelo alternativo reduce los falsos negativos un 85% y es más ligero — pero su licencia no permite uso comercial. Si el servicio es comercial, lo descartamos por licencia; si no lo es, es un candidato claro a considerar."*

**Nota:** esta sección **no modifica** los resultados del modelo actual (secciones 8–12). Las predicciones del alternativo se guardan en columnas separadas (`alt_label`, `alt_band`, `alt_score`) para poder compararlas sin alterar nada.